# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata  # This is an object, access as attributes
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# Explore the dataset's RecordSets, Fields, and Columns by their @id
# Note: mlcroissant provides access via dataset.metadata.record_sets. Each element has, e.g., .id and .fields
from pprint import pprint

print("# Available RecordSets:")
record_sets = dataset.metadata.record_sets
record_set_ids = []
for rs in record_sets:
    print(f"- RecordSet name: {rs.name}, @id: {rs.id}")
    record_set_ids.append(rs.id)
    print("  Fields/columns for this RecordSet:")
    for fld in rs.fields:
        # Each field corresponds to a column (fields have ids, names)
        print(f"    - Field: {fld.name}, @id: {fld.id}, type: {fld.data_type}")
    print("")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Extract data from each RecordSet
# You may select a subset, but here we load them all into pandas DataFrames
dataframes = {}
for record_set_id in record_set_ids:
    print(f"Loading records from RecordSet @id: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    if not records:
        print(f"  No records found in {record_set_id}")
        continue
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df

print("\n--- Columns for the first populated RecordSet ---")
# Find the first non-empty dataframe
first_df_id = None
for k, v in dataframes.items():
    if not v.empty:
        first_df_id = k
        break
if first_df_id:
    print(f"RecordSet @id: {first_df_id}")
    print(dataframes[first_df_id].columns.tolist())
    display(dataframes[first_df_id].head())
else:
    print("No RecordSet contains records.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# For EDA, pick a RecordSet and a relevant numeric and group field by their @id
# Modify these IDs/names according to your outputs above
"""
Example RecordSet and fields (replace as determined from previous cell):
- record_set_id: use e.g. 'cr:recordSet:proteins'
- numeric_field: '@id' of a numerical field, e.g. 'cr:field:Coverage(%)' or 'cr:field:PSMs'
- group_field: '@id' of a categorical field, e.g. 'cr:field:Sample'
"""
record_set_id = first_df_id  # Use the previously found populated RecordSet
if record_set_id is not None:
    df = dataframes[record_set_id]

    # Heuristically pick a numeric field - look for columns with float/int values
    numeric_field = None
    for col in df.columns:
        # Try to see if it looks numeric
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field = col
            break
    if numeric_field is None:
        print("No numeric field found for EDA.")
    else:
        print(f"Using numeric field: {numeric_field}")
        threshold = df[numeric_field].mean()  # Use the mean as an arbitrary threshold for demo
        filtered_df = df[df[numeric_field] > threshold].copy()
        print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
        display(filtered_df.head())

        # Normalization
        norm_field = f"{numeric_field}_normalized"
        filtered_df[norm_field] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        display(filtered_df[[numeric_field, norm_field]].head())

        # Try to find a group/categorical field
        group_field = None
        for col in df.columns:
            if pd.api.types.is_string_dtype(df[col]) and df[col].nunique() < 10 and col != numeric_field:
                group_field = col
                break
        if group_field:
            print(f"Grouping by field: {group_field}")
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            print(f"Grouped data by {group_field} (showing mean {numeric_field}):")
            display(grouped_df.head())
        else:
            print("No categorical/group field found for groupby analysis.")
else:
    print("No data available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if record_set_id is not None and numeric_field is not None:
    plt.figure(figsize=(8, 4))
    sns.histplot(data=df, x=numeric_field, bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field} in RecordSet {record_set_id}")
    plt.xlabel(numeric_field)
    plt.show()

    if group_field:
        plt.figure(figsize=(10,5))
        sns.boxplot(data=df, x=group_field, y=numeric_field)
        plt.title(f"{numeric_field} grouped by {group_field}")
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- Dataset contains information about protein abundance and properties from extracellular vesicles in human mast cells.
- The Croissant schema enables exploration of structure and field relationships via the `@id` for each entity.
- Standard EDA steps allow you to inspect numeric distributions, normalize values, and group data as needed for downstream analyses.

**You can now extend this notebook for deeper protein analysis, feature engineering, or statistical modeling.**